# Analise arquivo raw/fornecedores.csv

In [149]:
import os
import pandas as pd
from pandasql import sqldf

pysqldf = lambda q: sqldf(q, globals())

In [150]:
BASE_DIR = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath('.'))))
DATA_DIR = os.path.join(BASE_DIR, 'data')
RAW_DIR = os.path.join(DATA_DIR, 'raw')


In [151]:
df = pd.read_csv(os.path.join(RAW_DIR, 'fornecedores.csv'), sep=';')
df.shape

(26, 7)

## Analise exploratória

In [152]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 26 entries, 0 to 25
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id_fornecedor         26 non-null     str    
 1   nome_fornecedor       25 non-null     str    
 2   cidade                25 non-null     str    
 3   uf                    25 non-null     str    
 4   categoria             25 non-null     str    
 5   prazo_pagamento_dias  25 non-null     float64
 6   status                25 non-null     str    
dtypes: float64(1), str(6)
memory usage: 1.6 KB


In [153]:
df.head()

,id_fornecedor,nome_fornecedor,cidade,uf,categoria,prazo_pagamento_dias,status
0,F0001,Fornecedor Delta 01,Álvares Machado,SP,Bebidas,21.0,Ativo
1,F0002,Fornecedor Alfa 02,Presidente Prudente,SP,Tecnologia,21.0,Ativo
2,F0003,Fornecedor Alfa 03,Rancharia,SP,Serviços,60.0,ativo
3,F0004,Fornecedor Beta 04,presidente prudente,SP,Serviços,7.0,Ativo
4,F0005,Fornecedor Alfa 05,Regente Feijó,SP,Tecnologia,60.0,Ativo


In [154]:
df.tail()

,id_fornecedor,nome_fornecedor,cidade,uf,categoria,prazo_pagamento_dias,status
21,F0022,Fornecedor Delta 22,Santo Anastácio,SP,Serviços,7.0,Ativo
22,F0023,Fornecedor Alfa 23,Álvares Machado,SP,Serviços,7.0,Ativo
23,F0024,Fornecedor Omega 24,Álvares Machado,SP,Logística,14.0,Ativo
24,F0025,Fornecedor Gama 25,Presidente Bernardes,SP,Alimentos,21.0,Ativo
25,F0002,Fornecedor Alfa 02,Presidente Prudente,SP,Tecnologia,21.0,Ativo


In [155]:
df.sample(5)

,id_fornecedor,nome_fornecedor,cidade,uf,categoria,prazo_pagamento_dias,status
19,F0020,NaN,Santo Anastácio,SP,Embalagens,7.0,Ativo
17,F0018,Fornecedor Gama 18,Pirapozinho,SP,Tecnologia,45.0,NaN
15,F0016,Fornecedor Gama 16,Regente Feijó,SP,NaN,45.0,Ativo
18,F0019,Fornecedor Omega 19,Rancharia,SP,Embalagens,60.0,Ativo
14,F0015,Fornecedor Alfa 15,Martinópolis,SP,Tecnologia,7.0,Ativo


In [156]:
df.isna().sum()

id_fornecedor           0
nome_fornecedor         1
cidade                  1
uf                      1
categoria               1
prazo_pagamento_dias    1
status                  1
dtype: int64

In [157]:
df.columns

Index(['id_fornecedor', 'nome_fornecedor', 'cidade', 'uf', 'categoria',
       'prazo_pagamento_dias', 'status'],
      dtype='str')

In [158]:
q = '''SELECT nome_fornecedor
                , cidade
                , uf
                , categoria
                , prazo_pagamento_dias
                , status
        FROM df
        WHERE nome_fornecedor isnull
                or cidade isnull
                or uf isnull
                or categoria isnull
                or prazo_pagamento_dias isnull
                or status isnull

'''

In [159]:
print(pysqldf(q))

       nome_fornecedor           cidade   uf   categoria  \
0  Fornecedor Delta 06    Regente Feijó  NaN    Serviços   
1  Fornecedor Omega 11              NaN   SP   Alimentos   
2   Fornecedor Gama 16    Regente Feijó   SP         NaN   
3   Fornecedor Gama 18      Pirapozinho   SP  Tecnologia   
4                  NaN  Santo Anastácio   SP  Embalagens   

   prazo_pagamento_dias   status  
0                  14.0  Inativo  
1                   NaN    Ativo  
2                  45.0    Ativo  
3                  45.0      NaN  
4                   7.0    Ativo  


In [160]:
q = '''SELECT id_fornecedor
        FROM df
        WHERE nome_fornecedor isnull
                or cidade isnull
                or uf isnull
                or categoria isnull
                or prazo_pagamento_dias isnull
                or status isnull

'''

In [161]:
print(pysqldf(q))

  id_fornecedor
0         F0006
1         F0011
2         F0016
3         F0018
4         F0020


## Tratamento de dados

In [162]:
df_original = df.copy()

In [163]:
df.shape

(26, 7)

In [164]:
df = df.fillna('NAO INFORMADO')
df[['prazo_pagamento_dias']] = df[['prazo_pagamento_dias']].replace('NAO INFORMADO', 0)

In [165]:
df['prazo_pagamento_dias'] = pd.to_numeric(df['prazo_pagamento_dias'], errors='coerce')

In [166]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 26 entries, 0 to 25
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id_fornecedor         26 non-null     str    
 1   nome_fornecedor       26 non-null     str    
 2   cidade                26 non-null     str    
 3   uf                    26 non-null     str    
 4   categoria             26 non-null     str    
 5   prazo_pagamento_dias  26 non-null     float64
 6   status                26 non-null     str    
dtypes: float64(1), str(6)
memory usage: 1.6 KB


## Salvando dados Fornecedores em Bronze

In [167]:
BRONZE_DIR = os.path.join(DATA_DIR, 'bronze')

In [168]:
df.to_csv(os.path.join(BRONZE_DIR, 'b_fornecedores.csv'), index=False)